# Suite BRV — The Verilog bridge

`Bridges.Verilog` carries expressions between mbse-expressions' SystemVerilog dialect and the Verilog language's syntax
trees, both ways, tree to tree. Each case writes terms as syntax nodes, whose printed SystemVerilog is the dialect's
own render, and reads syntax nodes, parsed from SystemVerilog source, as terms. Constraints written as functions are
checked against the dialect's evaluator, and constraints from mbse-expressions' Basic dialect round-trip through it.

In [ ]:
import math

from mbse.Expressions.Dialects.SystemVerilog import Evaluators, Expressions as E, Text
from mbse.Expressions.Framework import Terms as F
from mbse.Programs.Bridges import Verilog as B
from mbse.Programs.Framework.Errors import TranspileError
from mbse.Programs.Verilog import SystemVerilog2023, Syntax as S
from support import raises


def printed(term):
    return SystemVerilog2023.print(B.expression(term))


def functions(*texts):
    """The functions that return the expressions `texts`, parsed from one source."""
    source = "".join(f"function automatic logic f{i}(); return {text}; endfunction\n" for i, text in enumerate(texts))
    return SystemVerilog2023.parse(source).items


def read(*texts):
    return [B.term_of_function(function) for function in functions(*texts)]


def fails(*cases):
    """Reading each (text, message)'s text fails with its message."""
    for function, (text, message) in zip(functions(*(text for text, _ in cases)), cases):
        with raises(TranspileError, match=message):
            B.term_of_function(function)

## BRV-01 · Every term of the dialect writes the SystemVerilog the dialect renders, and reads back the same

In [ ]:
x, y, z, q = (E.identifier(name) for name in "xyzq")
TERMS = [
    E.constant(5), E.vector("10x1", None, "b"), E.vector("11111111", True, "h"), E.vector("0101", None, "o"),
    E.vector("1100", None, "d"), E.vector("zzzz", None, "d"), E.unary("~", x), E.unary("!", E.binary("==", x, 1)),
    E.binary("-", x, E.binary("-", y, z)), E.binary("**", E.unary("-", x), 2), E.binary("->", x, E.binary("->", y, z)),
    E.binary("<->", E.binary("->", x, y), z), E.binary("==?", x, E.vector("1zz0", None, "b")), E.binary("<<<", x, 2),
    E.conditional(x, 1, 2), E.conditional(E.conditional(x, 1, 2), 3, 4), E.conditional(x, y, E.conditional(y, 1, 2)),
    E.concatenation(x, y, E.vector("01", None, "b")), E.replication(4, x), E.replication(2, E.concatenation(x, y)),
    E.select(x, 3), E.range_(x, 7, 0), E.select(E.member(x, "data"), E.binary("+", y, 1)),
    E.inside(x, 1, E.span(3, 5), y), E.binary("&&", E.inside(x, 1), y), E.unary("!", E.inside(x, 1)),
    E.cast("int", x), E.cast("signed", x), E.cast(8, x), E.cast("real", x), E.cast("logic", x), E.cast("unsigned", y),
    E.member(E.member(x, "a"), "b"), E.member(E.identifier("this"), "a"), E.call("$clog2", 8), E.call("f", x, 1),
    E.call("pkg.f"), E.call("this.f", x), E.method(q, "size"), E.method(q, "and"), E.method(E.member(x, "q"), "max"),
    E.iterate(q, "sum", "p", E.binary(">", E.identifier("p"), 0)), E.iterate(q, "xor", "item", E.identifier("item")),
    E.binary("+", E.method(q, "size"), 1), E.constant("a\"b\\c\n\t"), E.constant(2.5),
]
for term in TERMS:
    assert printed(term) == Text.ToText(term), (printed(term), Text.ToText(term))
for term, back in zip(TERMS, read(*(printed(term) for term in TERMS))):
    assert F.same(back, term), printed(term)
# A unary operator's operand is a primary (IEEE 1800, A.8.3): the printer parenthesizes a unary one, which the
# dialect's text does not.
NESTED = [(E.unary("-", E.unary("-", x)), "-(-x)"), (E.unary("&", E.unary("&", x)), "&(&x)")]
for (term, text), back in zip(NESTED, read(*(text for _, text in NESTED))):
    assert printed(term) == text and F.same(back, term), text
# A replication of several items repeats their concatenation.
assert F.same(read("{2{x, y}}")[0], E.replication(2, E.concatenation(x, y)))
assert isinstance(B.expression(E.identifier("this")), S.ThisExpression)  # SystemVerilog's keyword
with raises(TranspileError, match="a span is not an expression"):  # but an item of `inside`
    B.expression(E.span(1, 2))
print(len(TERMS), "terms")

## BRV-02 · Constants: an unsized decimal, a real or a string, a negative number a negation, and back

In [ ]:
CONSTANTS = [
    (0, "0"), (-1, "-1"), (2 ** 70, "1180591620717411303424"), (1.5, "1.5"), (-2.5, "-2.5"), (-0.0, "-0.0"),
    (1e16, "1e+16"), (1e-05, "1e-05"), (math.inf, "1.0 / 0.0"), (-math.inf, "-1.0 / 0.0"), (math.nan, "0.0 / 0.0"),
    ("a\"b\\c\n\t", '"a\\"b\\\\c\\n\\t"'), ("é", '"é"'), ("", '""'),
]
for (value, text), back in zip(CONSTANTS, read(*(text for _, text in CONSTANTS))):
    term = E.constant(value)
    assert printed(term) == text == Text.ToText(term), (printed(term), text)
    assert Text.ToText(back) == text, Text.ToText(back)  # a negation and a division read as written
assert F.same(read("-1")[0], E.unary("-", 1)) and F.same(read("1.0 / 0.0")[0], E.binary("/", 1.0, 0.0))
assert printed(E.binary("*", E.identifier("x"), -1)) == "x * -1"
print("ok")

## BRV-03 · Literals are decoded as IEEE 1800 reads them: sized literals' bits, extended or truncated, and escapes

In [ ]:
LITERALS = [
    ("8'b1?0z", "8'b00001z0z"), ("4'hx", "4'hx"), ("4'h1z", "4'hz"), ("3'hff", "3'h7"), ("12'o7x", "12'o007x"),
    ("8'sd255", "8'sd255"), ("8'dz", "8'dz"), ("4'D?", "4'dz"), ("2'd7", "2'd3"), ("16'h_ff_x", "16'h0ffx"),
    ("8'HFF", "8'hff"), ("4'sb1", "4'sb0001"), ("8'bx1", "8'bxxxxxxx1"), ("8'oz", "8'bzzzzzzzz"),
    ("12_345", "12345"), ("1.5e3", "1500.0"), ("1_0.5", "10.5"),
]
for (text, rendered), back in zip(LITERALS, read(*(text for text, _ in LITERALS))):
    assert Text.ToText(back) == rendered, (text, Text.ToText(back))
assert F.same(B.decode(S.IntegerLiteral(spelling="8'hFF")), E.vector("11111111", None, "h"))
assert B.decode(S.IntegerLiteral(spelling="8'sd255")).signed is True
STRINGS = [
    ('"\\101\\x41\\x4g"', "AA\x04g"), ('"\\n\\t\\\\\\"\\v\\f\\a"', "\n\t\\\"\v\f\a"), ('"\\q\\0"', "q\0"),
    ('"a\\\nb"', "ab"),
]
for (text, value), back in zip(STRINGS, read(*(text for text, _ in STRINGS))):
    assert back.value == value, (text, back.value)
# What slang refuses: a backslash that ends the text, and `\x` without digits.
for text, value in [("a\\", "a\\"), ("\\xg", "xg")]:
    assert B.decode(S.StringLiteral(text=text)).value == value, text
print("ok")

## BRV-04 · A constraint becomes a function, a `constraint` block or an assertion, and each reads back to it

In [ ]:
constraint = E.binary("&&", E.binary(">=", E.identifier("age"), 18), E.binary(">", E.method(E.identifier("email"), "size"), 0))
function = B.function_("is_contactable", [("int", "age"), ("byte", "email")], constraint)
assert SystemVerilog2023.print(function) == (
    "function automatic logic is_contactable(input int age, input byte email);\n"
    "    return age >= 18 && email.size() > 0;\nendfunction")
assert SystemVerilog2023.print(B.function_("half", [("real", "x")], E.binary("/", E.identifier("x"), 2.0), "real")) == (
    "function automatic real half(input real x);\n    return x / 2.0;\nendfunction")
assert SystemVerilog2023.print(B.constraint("c_age", constraint)) == "constraint c_age {\n    age >= 18 && email.size() > 0;\n}"
assert SystemVerilog2023.print(B.assertion(constraint)) == "assert (age >= 18 && email.size() > 0);"
assert SystemVerilog2023.print(B.assertion(constraint, "not contactable")) == (
    'assert (age >= 18 && email.size() > 0) else $error("not contactable");')
assert F.same(B.term_of_function(function), constraint) and F.same(B.term_of_constraint(B.constraint("c", constraint)), constraint)
assert F.same(B.term_of_assertion(B.assertion(constraint, "no")), constraint)
declared = SystemVerilog2023.parse("""class person;
    rand int age;
    // adults only
    constraint c_age { /* the constraint */ age >= 18 && age < 150; }
endclass
module m;
    initial assert (8'hff == 255) else $error("no");
    function automatic logic f(input int x);
        // the constraint
        return x inside {[1:3], 5};
    endfunction
endmodule
""")
assert Text.ToText(B.term_of_constraint(declared.items[0].items[2])) == "age >= 18 && age < 150"  # after the comment
assert Text.ToText(B.term_of_assertion(declared.items[1].items[0].body)) == "8'hff == 255"
assert Text.ToText(B.term_of_function(declared.items[1].items[1])) == "x inside {[1:3], 5}"
results = [Evaluators.OfAny(constraint, {"age": age, "email": email}) for age, email in [(20, [1]), (17, [1]), (30, [])]]
assert [result.integer() for result in results] == [1, 0, 0]  # the constraint, evaluated by the dialect
print("ok")

## BRV-05 · What the dialect cannot hold fails, at the syntax node's path

In [ ]:
ITERATION = "an iteration is a reduction (sum, product, and, or, xor) of one iterator"
CAST = "a cast names a built-in type, a signedness or a width"
fails(
    ("'b101", "body[0].value: an unsized based literal has no counterpart in the dialect"),
    ("'1 + 1", "body[0].value.left: UnbasedUnsizedLiteral has no counterpart in the SystemVerilog dialect"),
    ("x[3+:2]", "body[0].value: a part-select with +: has no counterpart in the dialect"),
    ("x[3-:2]", "body[0].value: a part-select with -: has no counterpart in the dialect"),
    ("x inside {[1 +/- 2]}", "body[0].value.set[0]: a tolerance range has no counterpart in the dialect"),
    ("my_t'(x)", f"body[0].value.type: {CAST}"), ("logic [3:0]'(x)", f"body[0].value.type: {CAST}"),
    ("int unsigned'(x)", f"body[0].value.type: {CAST}"), ("bit signed'(x)", f"body[0].value.type: {CAST}"),
    ("realtime'(x)", f"body[0].value.type: {CAST}"), ("const'(x)", f"body[0].value.type: {CAST}"),
    ("4'd8'(x)", f"body[0].value.type: {CAST}"),
    ("q.sum(p, i) with (p)", f"body[0].value: {ITERATION}"), ("q.sum with (item)", f"body[0].value: {ITERATION}"),
    ("q.min(p) with (p)", f"body[0].value: {ITERATION}"), ("q.sum(1) with (p)", f"body[0].value: {ITERATION}"),
    ("q.sum(p::x) with (p)", f"body[0].value: {ITERATION}"),
    ("p::f(1)", "body[0].value.callee: a function is called by its name"),
    ("$unit::x", "body[0].value: NameExpression has no counterpart in the SystemVerilog dialect"),
    ("f(.a(1))", "body[0].value.arguments[0]: NamedConnection has no counterpart in the SystemVerilog dialect"),
    ("$bits(int)", "body[0].value.arguments[0]: an argument is an expression"),
    ("f(, 1)", "body[0].value.arguments[0]: an argument is an expression"),
    ("x + (* a *) y", "body[0].value: attributes have no counterpart in the SystemVerilog dialect"),
    ("f (* a *) (1)", "body[0].value: attributes have no counterpart in the SystemVerilog dialect"),
    ("x ? x++ : y", "body[0].value.consequence: IncrementExpression has no counterpart in the SystemVerilog dialect"),
    ("1ns", "body[0].value: TimeLiteral has no counterpart in the SystemVerilog dialect"),
    ("'{1, 2}", "body[0].value: AssignmentPattern has no counterpart in the SystemVerilog dialect"),
    ("tagged a 5", "body[0].value: TaggedExpression has no counterpart in the SystemVerilog dialect"),
    ("super.a", "body[0].value.value: SuperExpression has no counterpart in the SystemVerilog dialect"),
)
one = S.IntegerLiteral(spelling="1")
with raises(TranspileError, match="the operator '++' has no counterpart in the SystemVerilog dialect"):
    B.term(S.UnaryExpression(operator="++", operand=one))
with raises(TranspileError, match="the operator 'x' has no counterpart in the SystemVerilog dialect"):
    B.term(S.BinaryExpression(left=one, operator="x", right=one))
print("ok")

## BRV-06 · The readers take what the writers make: one return, one expression constraint

In [ ]:
misfits = SystemVerilog2023.parse("""function automatic logic f(); endfunction
function automatic void g(); return; endfunction
function automatic int h(); int y = 1; return y; endfunction
function automatic int k(); k = 1; endfunction
class c; constraint a { x > 0; y > 0; } constraint b { soft x > 0; } constraint d { x -> y > 0; } constraint e { } endclass
module m; initial assert (1 == x[0+:2]); endmodule
""")
for function in misfits.items[:4]:
    with raises(TranspileError, match="the function's body must be one return of a value"):
        B.term_of_function(function)
for constraint in misfits.items[4].items:
    with raises(TranspileError, match="the constraint must hold one expression, not soft"):
        B.term_of_constraint(constraint)
with raises(TranspileError, match="expression.right: a part-select with +: has no counterpart in the dialect"):
    B.term_of_assertion(misfits.items[5].items[0].body)
print("ok")

## BRV-07 · Constraints round-trip through the translators, the bridge, the printer and the parser, and agree with Basic where it decides

In [ ]:
from mbse.Expressions import Evaluators as BasicEvaluators, Expressions as Basic, Translators
from mbse.Schemas.Framework import Proxies, Schemas as SC

Shipment = SC.OfObject.Builder().name("Shipment").ref().properties(
    lambda p: p.name("celsius").of(lambda t: t.as_native(float)), lambda p: p.name("hours").of(lambda t: t.as_native(int)),
    lambda p: p.name("readings").of(lambda t: t.as_indexed(lambda i: i.of(lambda n: n.as_native(float))))).create()
store = Proxies.OfStore()
store.register(Shipment)
shipments = [store.Shipment().celsius(5.5).hours(30).readings([4.0, 8.6, 9.1]).create(),
             store.Shipment().celsius(9.1).hours(80).readings([4.0, 12.5]).create(),
             store.Shipment().celsius(3.0).hours(20).readings([]).create()]  # no readings
this, c = Basic.variable("this"), Basic.variable("c")
CONSTRAINTS = [
    Basic.literal(2.0).le(this.celsius).and_(this.celsius.le(8.0)), this.hours.gt(72).implies(this.celsius.le(8.0)),
    this.hours.gt(72).not_(), this.readings.all("c", c.le(12.0)), this.readings.any("c", c.gt(8.0)),
    this.readings.count_where("c", c.gt(8.0)).le(2), this.readings.count().gt(0), this.readings.max().le(12.0),
    this.readings.item(0).ge(2.0), Basic.literal(4.0).in_(this.readings), this.hours.add(10).mul(2).sub(1).gt(100),
]
to_sv, to_basic = Translators.between(Basic.DIALECT, E.DIALECT), Translators.between(E.DIALECT, Basic.DIALECT)
terms = [to_sv.forward(constraint.data) for constraint in CONSTRAINTS]
sources = [SystemVerilog2023.print(B.function_(f"check{i}", [], term)) for i, term in enumerate(terms)]
checks = SystemVerilog2023.parse("\n".join(sources) + "\n")
for constraint, term, source, function in zip(CONSTRAINTS, terms, sources, checks.items):
    read_back = B.term_of_function(function)
    assert F.same(read_back, term) and F.same(to_basic.forward(read_back), constraint.data), source
    for shipment in shipments:  # where Basic decides, SystemVerilog's rules agree
        decided = BasicEvaluators.predicate(constraint, shipment)
        if decided is not None:
            assert Evaluators.OfAny(read_back, {"this": shipment}).integer() == decided, (source, decided)
assert sources[3] == (
    "function automatic logic check3;\n    return this.readings.and(c) with (c <= 12.0);\nendfunction")
print(len(CONSTRAINTS), "constraints")